# NeuraX — EEG Signal Exploration Notebook
This notebook demonstrates the full NeuraX pipeline:
1. Signal generation
2. Preprocessing & feature extraction
3. Model training & evaluation
4. Single prediction demo

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from data.signal_generator import generate_eeg_window, generate_dataset, COMMANDS
from preprocessing.pipeline import extract_features, bandpass_filter, preprocess_dataset
from models.classifier import train, predict, load_model

print('NeuraX notebook ready.')

## 1. Generate & Visualize EEG Signals

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Synthetic EEG Signals per Command', fontsize=14, fontweight='bold')

colors = ['#888', '#00d4ff', '#7b2fff', '#00ff88']
t = np.linspace(0, 1, 256)

for i, (cmd_id, cmd_name) in enumerate(COMMANDS.items()):
    sig = generate_eeg_window(cmd_id, noise_level=0.3)
    axes[i].plot(t, sig, color=colors[i], linewidth=0.8)
    axes[i].set_ylabel(cmd_name, fontsize=9)
    axes[i].set_facecolor('#010a11')
    axes[i].tick_params(colors='gray')
    axes[i].spines[:].set_color('#0d3a5c')

axes[-1].set_xlabel('Time (seconds)')
fig.patch.set_facecolor('#020b14')
plt.tight_layout()
plt.show()

## 2. Feature Extraction Demo

In [ ]:
raw = generate_eeg_window(command=1)  # cursor_left
features = extract_features(raw)

feature_names = ['delta','theta','alpha','beta','gamma','mean','std','var','skew','kurtosis','rms','peak']
print('Feature vector for cursor_left:')
for name, val in zip(feature_names, features):
    print(f'  {name:12s}: {val:.6f}')

## 3. Train Model

In [ ]:
model = train(samples_per_class=300)

## 4. Single Prediction

In [ ]:
for cmd_id in [0, 1, 2, 3]:
    signal = generate_eeg_window(cmd_id)
    result = predict(signal, model=model)
    actual = COMMANDS[cmd_id]
    predicted = result['command']
    conf = result['confidence'] * 100
    match = '✓' if actual == predicted else '✗'
    print(f'{match} Actual: {actual:14s}  Predicted: {predicted:14s}  Confidence: {conf:.1f}%')